# Anaylyse post preprocessing

## Objectif
L'objectif de cette preanaylyse est de ce rendre conpte des donnees en notre possessions et de leurs validite. 

### Etapes
Pour commencer on va verifier que notre column des témoignage contient des valeurs textuels et non des float, entier, ... 

In [1]:
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import numpy as np

In [1]:
import pandas as pd

try:
    df = pd.read_csv("../../Student_Dataset.csv") # Ouvre le csv
except FileNotFoundError:
    raise SystemExit(84)

dfStr = df.iloc[:, 1]

print(dfStr.dtypes)

str


In [2]:
print(df.iloc[:, 0].head(15))

2                                              0x000000
3                                              0x000000
4     A short sprint brought on a twinge in the midd...
5                                              0x000000
6     A sudden twist while reaching for a bag made o...
7                                              0x000000
8     I noticed a rash on my forearms after handling...
9                                              0x000000
10                                             0x000000
11    I feel an unexplained pressure behind my eyes ...
12                                             0x000000
13    There’s a small sore area on the top of my foo...
14    I have an increased thirst level and I’m going...
15                                             0x000000
16                                             0x000000
Name: 1, dtype: str


On voit donc que notre column numero deux qui contient les temoignage n'a que des string et aucun autre type

On va donc verifier les NaN potentiel

In [3]:
print(dfStr[dfStr.isna()])

4       NaN
6       NaN
8       NaN
11      NaN
13      NaN
       ... 
1000    NaN
1001    NaN
1003    NaN
1005    NaN
1010    NaN
Name: On uneven paths the heel feels wobbly side‑to‑side. Slanted pavements bring a quick aching shift under the ankle when I step., Length: 396, dtype: str


On comprend un probleme, comme nous n'avons pas de label la column 2 se melange avec `0x000000` et le temoignage.
Pour aller plus loins dans l'anaylse va donc retirer tout les `0x000000`, ils n'apportent rien aux temoignage, on va dnc le retirer pour avoir une deuxieme column parfaitement conforme et savoir si les temoignange contiennet des NaN.

In [4]:
dfStr = dfStr.str.replace('0x000000,', '', regex=False)
print(dfStr)

2       The feverish feeling eased, but I’m now breath...
3       While working in the garden, I touched a metal...
4                                                     NaN
5       I accidentally left my hand on a hot mug for a...
6                                                     NaN
                              ...                        
1007    I slammed my fingertip in a drawer. The tip is...
1008    I woke up with a throbbing pain in my left ear...
1009    During the summer, my throat gets itchy and I ...
1010                                                  NaN
1011    Every morning I wake up with a pounding headac...
Name: On uneven paths the heel feels wobbly side‑to‑side. Slanted pavements bring a quick aching shift under the ankle when I step., Length: 1010, dtype: str


In [5]:
df = pd.read_csv('../../Student_Dataset.csv')

# Si la colonne 1 est vide, le texte est dans la column 0. SInon on prend le 1
textes_propres = np.where(df.iloc[:, 1].isna(), df.iloc[:, 0], df.iloc[:, 1])

# on genere un tableau clean avec deux column
df_final = pd.DataFrame({
    'Chiffre': range(1, len(df) + 1),
    'Temoignage': textes_propres
})


print(df_final.head(10))

   Chiffre                                         Temoignage
0        1  The feverish feeling eased, but I’m now breath...
1        2  While working in the garden, I touched a metal...
2        3  A short sprint brought on a twinge in the midd...
3        4  I accidentally left my hand on a hot mug for a...
4        5  A sudden twist while reaching for a bag made o...
5        6  During a birthday party, a hot beverage was kn...
6        7  I noticed a rash on my forearms after handling...
7        8  Since I lost 20 pounds, my headaches have less...
8        9  The rash on my face and jaw is a classic hives...
9       10  I feel an unexplained pressure behind my eyes ...


On vient donc de regler nos problemes de column et de data, on a donc cree un nouveau tableau propre qui ne contient plus les `0x000000`. De plus nous avons ajoute des label pour plus de simplicite.
On peut donc maintenant verifier si un NaN est present.

In [6]:
print(df_final.isna().sum())

Chiffre       0
Temoignage    0
dtype: int64


On obtient donc 0 dans chiffre et temoignage donc nous n'avons aucune valeur NaN

Nous allons faire une derniere verification, si le temoignage est inferieur a 10 caracteres on va considerer que le temoignange n'est pas complet/ pertinent car trop petit

In [7]:
df_final = df_final[df_final['Temoignage'].str.len() > 10]

Nous avons donc nettoyer les temoignages de moins de 10 caracteres.

##### Verification du nombre de doublon

In [8]:
print("Nombre de témoignages en double :", df_final.duplicated(subset=["Temoignage"]).sum())

Nombre de témoignages en double : 9


Nous observons 15 doublons. Nous n'avons pas la possibilite de savoir si ces temoignage sont des duplications ou simpelement des temoignage identique.
Poour garder l'integrite du dataset nous allons les supprimer, car ils ne representent qu'une infime partie.

Après réalisation de graphique, nous avons des apostrophes qui ne sont pas sous forme classique, nous allons donc les changer en `'` pour normaliser le texte.

In [9]:
df_final['Temoignage'] = df_final['Temoignage'].str.replace('’', "'")
raise SystemExit(0)

Les apostrophe ont bien etaient changes avec succes.
L'analyse du CSV est donc termine, nous avons toute les informations necessaire pour realiser notre nouveau CSV propre ()

#### On va regarder les images similaires 
Notre dataset est unr egroupement de plusieurs scanner d'hopitaux au vu des resultats sur les premiers model il va falloir verifier dans le `.npy` si nous n'avons pas des images dupliquer qui ont une matrice similaire.



In [15]:
# on charge notre fichier de matrice
matrices = np.load('../../Matrice.npy', allow_pickle=True)

# on applati les matrice pour les comparer + facilement
flat = matrices.reshape(matrices.shape[0], -1)

# mesure la direction entre deux valeurs (images) et donne un chiffre de similitude
sim = cosine_similarity(flat)
# on trouve les similaire mais on prend l'image elle meme
np.fill_diagonal(sim, 0)

prctRessemblance = 0.995
paris_P_Identique = np.argwhere(sim > prctRessemblance)

print(f"Nombre de paires quasi identique : {len(paris_P_Identique)//2}")

Nombre de paires quasi identique : 6142


On charge notre matriuce et on fait un cosine similarity pour savoir combien d'images ont un doublon.

On doit ensuite etablir les groupes d'image similaire.

In [14]:
import numpy as np

def find(parent, i):
        """
        Permet de remonter dans tout les parents jusqu'a se trouver elle meme.
        Si on a [0, 0, 1, 4]
        on cherche find(parent, 1), on obtiens 0 donc on fait find(parent, 0) et on obteiens 0 (la meme image que le i) donc on a termine on est retombe desus, alors 1 et 0 sont les meme

        Prend en argument parent et i:
            - i est l'image qu'on cherche
            - parent est une liste qui contient globalement qui pointe vers qui, imaginons image A pointe vers l'image B alors dans parent on aura B, et ca se cumule avec les autres enfants

        return i
        """
        # tant qu'one est pas sur le meme
        while parent[i] != i:
            i = parent[i]
        return i

def union(parent, i, j):
    """
    Prend en parametre parent i et j:
        - parent est la liste qui contient qui pointe vers qui
        - i et j sont les images qu'on veut fusionner
    """
    lst_i = find(parent, i)
    list_j = find(parent, j)
    # on verif qu'ils sont pas deja dans le meme groupe
    if lst_i != list_j:
        # on fait pointer i vers j qui devient son parent
        parent[list_j] = lst_i


# on prend la premiere column
n = sim.shape[0]

# creer le tableau parent de base, au debut chaque image pointe vers elle meme car elel est dientique a elle meme
parent = np.arange(n)


# on va faire des paris d'images similaire
pairs = np.argwhere(sim > prctRessemblance)
# (voir en bas pour l'explication)

# on passe dans chacune des pairs pour trouver les unions avec les autres paris
for i, j in pairs:
    union(parent, i, j)


# on va stoccker tt les resultats 
resultat = []
for i in range(n):
    resultat.append(find(parent, i))


nbr_groupes = len(np.unique(resultat))
print(f"Nombre de groupes donc patients : {nbr_groupes}")
print(f"Taille moyenne d'1 groupe : {n / nbr_groupes:.1f} images")

Nombre de groupes donc patients : 6885
Taille moyenne d'1 groupe : 1.8 images


##### Explication de argwhere et du system actuel pour les images similaire

Noytre systeme se base sur les matrice, imaginons que nous avons au total 4 images :

```
    [0.0,  0.99, 0.2,  0.998],
    [0.99, 0.0,  0.1,  0.3],
    [0.2,  0.1,  0.0,  0.15],
    [0.998,0.3,  0.15, 0.0]
```

on a donc une matrice de 4x4 avec des 0 en diagonale car on a dit `np.fill_diagonal(sim, 0)` donc column 1 ligne 1 qui est l'image 1 ne peut pas etre identique a 1.

En suite on fait sim > prctRessemblance donc on passe dans chacun et on regarde si la valeur est > a 0.995.
Prenons column 1 ligne 0 on a donc 0.99 donc c'est < a 0.995 alors on va mettre false a la palce.

On fait ca pour tou et on obtiens :

```
    [False, False, False, True],
    [False, False, False, False],
    [False, False, False, False],
    [True,  False, False, False]
```

Donc on a ligne 0 column 4 et column 0 ligne 4, nous avons donc trouver une pair [0, 4] donc ces deux images seront similaire 


On passe ensutie dans les pairs et on essaye de trouver les imilaires et par ricochet ca nous donnera toute les similaires.


##### Pourquoi faire ca ?

COmme notre dataset contient du dataleak car le dataset contient plusieurs scanners d'hopitaux, certaines personnes ont fait plusieurs scanner dans le meme ou un different hopital.
IL ya  donc des images du meme patient approximativement 1.8 par patient (moyenne d'un groupe).

DOnc pour eviter de fausser nos donnes on ne vas pas inclure pour un meme patient une de ces images dans l'entraienemtn et 1 dans le test car sinon il va trouver directement sans chercher, il repondra car il l'a appris pas coeur.
DOnc on envoie toute les images d'un patient dans 1 seule des deux aprtie seulement (entrainement ou test) et c'est tout.